In [25]:
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

In [26]:
def charger_immobilier():
    """Charge California Housing, renvoie X, y.
    Affiche de manière robuste le nombre de lignes, de variables et l'unité de la cible.
    """
    data = fetch_california_housing()
    X, y = data.data, data.target
    
    nb_lignes, nb_variables = X.shape
    nom_cible = data.target_names[0] if hasattr(data, 'target_names') else "Cible"
    
    # Extraction robuste de la ligne descriptive de la cible
    unite_cible = "Non trouvée textuellement"
    
    for line in data.DESCR.split('\n'):
        # On cherche des mots-clés universels dans la description du dataset
        if "median house value" in line.lower() or nom_cible.lower() in line.lower():
            if ":" in line: # Souvent formaté comme "Nom : description"
                unite_cible = line.split(":", 1)[1].strip()
            else:
                unite_cible = line.strip()
            break

    # Affichage propre
    print(f"California Housing : ({nb_lignes}, {nb_variables}) variables")
    print(f"Cible ({nom_cible}) : {unite_cible}")
    
    return X, y

In [27]:
X, y = charger_immobilier()
    
# Séparation Train/Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardisation
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)



California Housing : (20640, 8) variables
Cible (MedHouseVal) : The target variable is the median house value for California districts,


In [28]:
def evaluer_regression(modele, X_train, X_test, y_train, y_test):
    """Entraîne, prédit, renvoie un dict {r2, mae, rmse}.
    Doit renvoyer les 3 métriques de régression vues en section 2.
    """
    # Entraînement
    modele.fit(X_train, y_train)
    
    # Prédiction
    y_pred = modele.predict(X_test)
    
    # Calcul des métriques
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = root_mean_squared_error(y_test, y_pred)
    
    return {"r2": r2, "mae": mae, "rmse": rmse}

In [29]:
# Définition des modèles
lr = LinearRegression()
rf = RandomForestRegressor(random_state=42, n_estimators=100)

In [30]:

# ------------------------------------------
# CAS NORMAL : Dataset complet
# ------------------------------------------
print("\n--- CAS NORMAL ---")
res_lr = evaluer_regression(lr, X_train_scaled, X_test_scaled, y_train, y_test)
res_rf = evaluer_regression(rf, X_train_scaled, X_test_scaled, y_train, y_test)

print(f"LinearRegression : R2={res_lr['r2']:.2f} MAE={res_lr['mae']:.2f} RMSE={res_lr['rmse']:.2f}")
print(f"RandomForest     : R2={res_rf['r2']:.2f} MAE={res_rf['mae']:.2f} RMSE={res_rf['rmse']:.2f}")


--- CAS NORMAL ---
LinearRegression : R2=0.58 MAE=0.53 RMSE=0.75
RandomForest     : R2=0.80 MAE=0.33 RMSE=0.51


In [31]:
# ------------------------------------------
# CAS LIMITE : Entraînement sur 100 lignes
# ------------------------------------------
print("\n--- CAS LIMITE (100 lignes) ---")
X_train_lim, y_train_lim = X_train_scaled[:100], y_train[:100]

res_lr_lim = evaluer_regression(lr, X_train_lim, X_test_scaled, y_train_lim, y_test)
res_rf_lim = evaluer_regression(rf, X_train_lim, X_test_scaled, y_train_lim, y_test)

print(f"LinearRegression (Limité) : R2={res_lr_lim['r2']:.2f} MAE={res_lr_lim['mae']:.2f} RMSE={res_lr_lim['rmse']:.2f}")
print(f"RandomForest (Limité)     : R2={res_rf_lim['r2']:.2f} MAE={res_rf_lim['mae']:.2f} RMSE={res_rf_lim['rmse']:.2f}")


--- CAS LIMITE (100 lignes) ---
LinearRegression (Limité) : R2=0.40 MAE=0.54 RMSE=0.88
RandomForest (Limité)     : R2=0.54 MAE=0.56 RMSE=0.77


In [32]:
# ------------------------------------------
# CAS ADVERSARIAL : Quartier fictif hors-norme
# ------------------------------------------
print("\n--- CAS ADVERSARIAL ---")
valeurs_moyennes = np.mean(X, axis=0)
quartier_fictif = valeurs_moyennes.copy()
quartier_fictif[0] = 0.0    # Revenu médian à 0
quartier_fictif[4] = 9000.0 # Population énorme

quartier_fictif_scaled = scaler.transform([quartier_fictif])

pred_lr = lr.predict(quartier_fictif_scaled)[0]
pred_rf = rf.predict(quartier_fictif_scaled)[0]

print(f"Prédiction LinearRegression : {pred_lr:.2f} (soit {pred_lr*100000:.0f} $)")
print(f"Prédiction RandomForest     : {pred_rf:.2f} (soit {pred_rf*100000:.0f} $)")


--- CAS ADVERSARIAL ---
Prédiction LinearRegression : 0.17 (soit 16913 $)
Prédiction RandomForest     : 1.19 (soit 119458 $)


In [33]:
def charger_airbnb(url_csv):
    """Charge le CSV, garde les colonnes numériques utiles, nettoie les NaN.

    Doit renvoyer un DataFrame propre, sans valeurs manquantes.
    """
    # 1. Chargement des données
    df = pd.read_csv(url_csv)

    # 2. Sélection des colonnes numériques pertinentes
    colonnes_utiles = [
        "price",
        "minimum_nights",
        "number_of_reviews",
        "availability_365",
    ]

    # Sécurité au cas où les noms de colonnes varient légèrement selon le dataset d'Airbnb
    colonnes_presentes = [col for col in colonnes_utiles if col in df.columns]
    df_numeric = df[colonnes_presentes]

    df
    # 3. Gestion des NaN (Imputation par la médiane pour ne pas perdre de lignes, ou dropna)
    # Ici, on opte pour dropna pour avoir des données 100% réelles, ou fillna pour la robustesse.
    df_clean = df_numeric.dropna().copy()
    df_clean["price"] = df_clean["price"].str.replace("$", "", regex=False).str.replace(",", "", regex=False).astype(float)
    print(
        f"Listings chargés : {df_clean.shape[0]} lignes, {df_clean.shape[1]} colonnes numériques retenues"
    )
    return df_clean

In [44]:
df_airbnb = charger_airbnb("listings.csv")

Listings chargés : 5256 lignes, 4 colonnes numériques retenues


In [40]:
def choisir_k(X_scaled, k_range=range(2, 15)):
    """Pour chaque k, renvoie inertie et silhouette.

    Doit afficher un tableau permettant de repérer le coude / le meilleur k.
    """
    print(f"{'k':<5} | {'Inertie':<12} | {'Silhouette':<10}")
    print("-" * 35)

    for k in k_range:
        # Initialisation de KMeans (n_init=10 explicite pour éviter les warnings)
        kmeans = KMeans(n_clusters=k, n_init=10, random_state=42)
        cluster_labels = kmeans.fit_predict(X_scaled)

        # Récupération des métriques
        inertia = kmeans.inertia_
        sil = silhouette_score(X_scaled, cluster_labels)

        # Affichage formaté
        print(f"k={k:<3} | inertie={inertia:<10.1f} | silhouette={sil:.2f}")

In [45]:
# Standardisation
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_airbnb)

choisir_k(X_scaled)

k     | Inertie      | Silhouette
-----------------------------------
k=2   | inertie=15864.0    | silhouette=0.76
k=3   | inertie=11647.4    | silhouette=0.38
k=4   | inertie=8484.4     | silhouette=0.43
k=5   | inertie=6053.0     | silhouette=0.46
k=6   | inertie=4946.9     | silhouette=0.47
k=7   | inertie=4183.0     | silhouette=0.47
k=8   | inertie=3805.5     | silhouette=0.34
k=9   | inertie=3471.1     | silhouette=0.35
k=10  | inertie=3166.1     | silhouette=0.35
k=11  | inertie=2871.4     | silhouette=0.36
k=12  | inertie=2613.2     | silhouette=0.36
k=13  | inertie=2459.7     | silhouette=0.32
k=14  | inertie=2266.2     | silhouette=0.32


In [ ]:
print("--- 1. CAS NORMAL ---")


# Entraînement du modèle final
k_choisi = 6
kmeans_normal = KMeans(n_clusters=k_choisi, n_init=10, random_state=42)
df_airbnb["cluster_normal"] = kmeans_normal.fit_predict(X_scaled)

# Description des segments basée sur les moyennes des vraies données
print("\nProfil des segments (Cas Normal) :")
print(df_airbnb.groupby("cluster_normal").mean())



--- 1. CAS NORMAL ---

Profil des segments (Cas Normal) :
                      price  minimum_nights  number_of_reviews  \
cluster_normal                                                   
0                 87.897941        5.897065          40.315375   
1                130.470588      364.235294          23.223529   
2                 86.270959        3.525243          40.980083   
3                 81.528217        2.004515         381.613995   
4               1213.111111        3.833333          17.611111   
5                340.044776        2.992537          36.660448   

                availability_365  
cluster_normal                    
0                     300.604906  
1                     333.682353  
2                      77.735063  
3                     208.316027  
4                     288.000000  
5                     231.690299  


Cluster 0 : Les "Dispo Long Terme / Standard"

Profil : Prix modéré (88 €), séjours d'environ une semaine (~6 nuits), moyennement commentés, mais disponibles presque toute l'année (300 jours).

Cluster 1 : Les "Baux Commerciaux / Professionnels"

Profil : Ce groupe est très marqué par un minimum_nights à 364 nuits. Ce ne sont pas des locations de vacances, mais des appartements meublés loués à l'année (étudiants, professionnels en mobilité).

Cluster 2 : Les "Occasionnels / Locaux"

Profil : Prix abordable (86 €), séjours courts (3-4 nuits). La particularité est leur très faible disponibilité (77 jours par an). Ce sont probablement des locaux qui louent leur propre logement uniquement pendant leurs vacances.

Cluster 3 : Les "Stars de la plateforme / Ultra-populaires"

Profil : Un nombre d'avis gigantesque (381 avis en moyenne) pour un prix attractif (81 €) et des séjours courts (2 nuits). Ce sont des logements à forte rotation (ex: petits studios hyper-centraux ou proches des gares) gérés de manière industrielle.

Cluster 4 : Le "Grand Luxe / Ultra-Premium"

Profil : Prix astronomique (1 213 € la nuit en moyenne), peu d'avis. Ce segment regroupe les villas d'exception ou les penthouses de luxe.

Cluster 5 : Le "Premium Standard"

Profil : Logements haut de gamme mais accessibles (340 € la nuit), loués pour des week-ends ou courts séjours (3 nuits).

In [46]:
# =====================================================================
# 2. CAS LIMITE : SANS STANDARDISATION
# =====================================================================
print("\n--- 2. CAS LIMITE (SANS STANDARDISATION) ---")
kmeans_sans_scale = KMeans(n_clusters=k_choisi, n_init=10, random_state=42)
df_airbnb["cluster_sans_scale"] = kmeans_sans_scale.fit_predict(df_airbnb[
    ["price", "minimum_nights", "number_of_reviews", "availability_365"]
])

print("\nProfil des segments (Sans Standardisation) :")
print(df_airbnb.groupby("cluster_sans_scale").mean())




--- 2. CAS LIMITE (SANS STANDARDISATION) ---

Profil des segments (Sans Standardisation) :
                          price  minimum_nights  number_of_reviews  \
cluster_sans_scale                                                   
0                    349.486056       15.976096          34.366534   
1                     89.239051       17.437500          29.093522   
2                     84.320652        2.882246         241.692029   
3                   1213.111111        3.833333          17.611111   
4                     80.922535        4.154930         578.767606   
5                     87.529748        4.431223          34.922418   

                    availability_365  
cluster_sans_scale                    
0                         242.462151  
1                         304.424726  
2                         214.331522  
3                         288.000000  
4                         209.661972  
5                          76.445026  


In [49]:
# =====================================================================
# 3. CAS ADVERSARIAL : INJECTION D'UN OUTLIER
# =====================================================================
print("\n--- 3. CAS ADVERSARIAL (VALEUR ABERRANTE) ---")

# On crée une copie des données chargées 
df_adversarial = df_airbnb[
    ["price", "minimum_nights", "number_of_reviews", "availability_365"]
].copy()

# On lance le KMeans
kmeans_adv = KMeans(n_clusters=k_choisi, n_init=10, random_state=42)
df_adversarial["cluster_adv"] = kmeans_adv.fit_predict(X_scaled)

print("\nTaille des clusters avant injection de l'outlier :")
print(df_adversarial["cluster_adv"].value_counts())

df_adversarial_fake = df_airbnb[
    ["price", "minimum_nights", "number_of_reviews", "availability_365"]
].copy()
# on injecte l'annonce à 100 000 €
annonce_fake = pd.DataFrame(
    [{
        "price": 100000.0,
        "minimum_nights": 1,
        "number_of_reviews": 0,
        "availability_365": 365,
    }]
)
df_adversarial_fake = pd.concat([df_adversarial_fake, annonce_fake], ignore_index=True)

# On doit re-standardiser ce nouveau dataset pollué
X_adv_scaled = scaler.fit_transform(df_adversarial_fake)

# On relance le KMeans
kmeans_adv = KMeans(n_clusters=k_choisi, n_init=10, random_state=42)
df_adversarial_fake["cluster_adv"] = kmeans_adv.fit_predict(X_adv_scaled)

print("\nTaille des clusters après injection de l'outlier :")
print(df_adversarial_fake["cluster_adv"].value_counts())




--- 3. CAS ADVERSARIAL (VALEUR ABERRANTE) ---

Taille des clusters avant injection de l'outlier :
cluster_adv
0    2283
2    2159
3     443
5     268
1      85
4      18
Name: count, dtype: int64

Taille des clusters après injection de l'outlier :
cluster_adv
4    2285
1    2182
0     563
5     141
2      85
3       1
Name: count, dtype: int64


In [51]:
print("\nProfil des centres de clusters normaux :")
print(df_adversarial.groupby("cluster_adv").mean())
print("\nProfil des centres de clusters pollués (Données brutes) :")
print(df_adversarial_fake.groupby("cluster_adv").mean())


Profil des centres de clusters normaux :
                   price  minimum_nights  number_of_reviews  availability_365
cluster_adv                                                                  
0              87.897941        5.897065          40.315375        300.604906
1             130.470588      364.235294          23.223529        333.682353
2              86.270959        3.525243          40.980083         77.735063
3              81.528217        2.004515         381.613995        208.316027
4            1213.111111        3.833333          17.611111        288.000000
5             340.044776        2.992537          36.660448        231.690299

Profil des centres de clusters pollués (Données brutes) :
                     price  minimum_nights  number_of_reviews  \
cluster_adv                                                     
0                88.916519        2.209591         241.120782   
1                99.367553        3.531164          34.458295   
2              